In [385]:
import matplotlib.pyplot as plt
import numpy as np
import random
from IPython.display import Image
from qutip import (Qobj, tensor, basis, create, destroy, qeye, mesolve, steadystate, expect, num)

In [ ]:
##########################################
# CONSTANTS, BASIS STATES, AND OPERATORS #
##########################################

# Constants
# TODO - GET ACTUAL VALUES!!!
hbar = 1
epsilon_0 = 1
c = 1
boltzmann_const = 1
Rb_mass = 1
pressure = 1
atomic_density = lambda temp: pressure / (boltzmann_const * temp)
kappa = 1.0 # Write all parameters in terms of kappa
Gamma_12 = 0.1 * kappa
Gamma_13 = 0.1 * kappa
Gamma_24 = 0.1 * kappa
Gamma_34 = 0.1 * kappa
N_Fock = 3 # Truncate each Fock basis to N_Fock photons
d_13 = 1
d_24 = 1
omega_13 = 1 / 795
omega_24 = 1 / 1530

# Atomic Basis Kets
atomicKet1 = basis(4, 0) # |1>
atomicKet2 = basis(4, 1) # |2>
atomicKet3 = basis(4, 2) # |3>
atomicKet4 = basis(4, 3) # |4>


# Set up operators in the relevant basis
S11 = tensor(atomicKet1 * atomicKet1.dag(), qeye(N_Fock), qeye(N_Fock))
S22 = tensor(atomicKet2 * atomicKet2.dag(), qeye(N_Fock), qeye(N_Fock))
S33 = tensor(atomicKet3 * atomicKet3.dag(), qeye(N_Fock), qeye(N_Fock))
S44 = tensor(atomicKet4 * atomicKet4.dag(), qeye(N_Fock), qeye(N_Fock))

S12 = tensor(atomicKet1 * atomicKet2.dag(), qeye(N_Fock), qeye(N_Fock)) # Photon emission 1->2
S13 = tensor(atomicKet1 * atomicKet3.dag(), qeye(N_Fock), qeye(N_Fock)) # Photon emission 1->3
S24 = tensor(atomicKet2 * atomicKet4.dag(), qeye(N_Fock), qeye(N_Fock)) # Photon emission 2->4
S34 = tensor(atomicKet3 * atomicKet4.dag(), qeye(N_Fock), qeye(N_Fock)) # Photon emission 3->4

S21 = S12.dag() # Photon absorption 2->1
S31 = S13.dag() # Photon absorption 3->1
S42 = S24.dag() # Photon absorption 4->2
S43 = S34.dag() # Photon absorption 4->3

# a and b act on photon independent Fock bases as lowering operator
a = tensor(qeye(4), destroy(N_Fock), qeye(N_Fock))
b = tensor(qeye(4), qeye(N_Fock),    destroy(N_Fock))

# Creation operators
a_dag = a.dag()
b_dag = b.dag()

# Number Operator for field a
num_a_op = tensor(qeye(4), num(N_Fock),  qeye(N_Fock))
num_b_op = tensor(qeye(4), qeye(N_Fock), num(N_Fock))

# Decay operators
# TODO - These may be temperature dependent!!
cavity_a_decay = np.sqrt(2*kappa) * a
cavity_b_decay = np.sqrt(2*kappa) * a
atomic_decay_12 = np.sqrt(2*Gamma_12) * S12
atomic_decay_13 = np.sqrt(2*Gamma_13) * S13
atomic_decay_24 = np.sqrt(2*Gamma_24) * S24
atomic_decay_34 = np.sqrt(2*Gamma_34) * S34
c_ops = [
    cavity_a_decay,
    cavity_b_decay,
    atomic_decay_12,
    atomic_decay_13,
    atomic_decay_24,
    atomic_decay_34
]

In [387]:
###############
# HAMILTONIAN #
###############

def form_full_Hamiltonian(
        Omega_s,  # Signal Rabi Frequency (Related to electric field strength)
        Omega_i,  # Idler Rabi Frequency (Related to electric field strength)
        Omega_I,  # Pump 1 Rabi Frequency (Related to electric field strength)
        Omega_II, # Pump 2 Rabi Frequency (Related to electric field strength)
        Delta_s,  # Signal detuning
        Delta_i,  # Idler detuning
        Delta_I,  # Pump 1 detuning
        Delta_II  # Pump 2 detuning
    ):
    # Pseudocode Hamiltonian:
    # H = hbar[
    #       0,               Omega_I,         Omega_S*a_dag,  0
    #       conj(Omega_I),   Delta_I,         0,              Omega_I*b_dag
    #       conj(Omega_s)*a, 0,               Delta_s,        Omega_II
    #       0,               conj(Omega_I)*b, conj(Omega_II), Delta_s + Delta_II
    #     ] + hbar*Delta_s*a_dag*a + hbar*Delta_i*b_dag*b

    H_atom = hbar * (
        Delta_I * S22 +
        Delta_s * S33 + 
        (Delta_II + Delta_s) * S44
    )

    H_photon = hbar * (
        Delta_s * a_dag * a +
        Delta_i * b_dag * b
    )

    H_int = hbar * (
        Omega_I  * S12         + np.conj(Omega_I)  * S21     +
        Omega_s  * a_dag * S13 + np.conj(Omega_s)  * a * S31 +
        Omega_i  * b_dag * S24 + np.conj(Omega_i)  * b * S42 +
        Omega_II * S34         + np.conj(Omega_II) * S43
    )

    H_total = H_atom + H_photon + H_int
    return H_total




In [ ]:
def Maxwell_Bloch_step(rho, Omega_s, Omega_i, Delta_s, Delta_i, temp, step_size):
    rho_13 = rho[1,3]
    rho_24 = rho[2,4]
    # TODO - CHECK THESE FORMULAS
    d_z_Omega_s = 1j * (omega_13 + Delta_s) * d_13**2 * atomic_density(temp) * rho_13 / (2 * epsilon_0 * hbar)
    d_z_Omega_i = 1j * (omega_24 + Delta_i) * d_24**2 * atomic_density(temp) * rho_24 / (2 * epsilon_0 * hbar)

    Omega_s_step = d_z_Omega_s * step_size
    Omega_i_step = d_z_Omega_i * step_size

    # Update rabie frequencies
    new_Omega_s = Omega_s + Omega_s_step
    new_Omega_i = Omega_i + Omega_i_step

    return new_Omega_s, new_Omega_i




def solve_for_steady_state_doppler_broadened_density_matrix(
        Omega_s,         # Signal Rabi Frequency (Related to electric field strength)
        Omega_i,         # Idler Rabi Frequency (Related to electric field strength)
        Omega_I,         # Pump 1 Rabi Frequency (Related to electric field strength)
        Omega_II,        # Pump 2 Rabi Frequency (Related to electric field strength)
        Delta_s,         # Signal detuning
        Delta_i,         # Idler detuning
        Delta_I,         # Pump 1 detuning
        Delta_II,        # Pump 2 detuning
        temp,            # Temperature of Rb gas
        num_velocity_subclasses = 20
    ):

    # Set up normal velocity distribution
    sigma_sq = 2 * boltzmann_const * temp / Rb_mass
    maxwell_boltzmann_dist = lambda v: 1 / np.sqrt(np.pi * sigma_sq) * np.exp(-v**2 / sigma_sq)

    # Range of velocities to sample over (2 standard deviations)
    min_v = -2 * np.sqrt(sigma_sq)
    max_v = 2 * np.sqrt(sigma_sq)
    v_vals = np.linspace(min_v, max_v, num_velocity_subclasses)

    rho_sum = None

    for v in v_vals:
        # TODO - Update to be + k*v (for all 4 k values)
        # NOTE - Each k value is a function of the index of diffraction, which is a
        #        function of driving frequency, so we may need to solve numerically here
        # TODO - Make this a python function such that it can be updated and changed
        #
        ###########################
        # NOTE - IMPORTANT - NOTE #
        ###########################
        # It may be the case that by using a certain arrangement of copropagating beams,
        # the effect of doppler broadening could be cancelled out.
        # DO THE MATH TO SEE IF THIS IS POSSIBLE!
        doppler_shift = (c + v) / c

        # Solve for the steady state
        H_total = form_full_Hamiltonian(
            Omega_s, Omega_i, Omega_I, Omega_II,
            Delta_s * doppler_shift,
            Delta_i * doppler_shift,
            Delta_I * doppler_shift,
            Delta_II * doppler_shift
        )
        rho = steadystate(H_total, c_ops, method='direct')

        weight = maxwell_boltzmann_dist(v)

        if rho_sum is None:
            rho_sum = weight * rho
        else:
            rho_sum += weight * rho
        
    # The resulting rho_sum should be approximately normalized,
    # but fix any discrepancy here
    doppler_broadened_rho = rho_sum.unit()

    return doppler_broadened_rho
    

    

def iterative_solve(
        Omega_s,         # Signal Rabi Frequency (Related to electric field strength)
        Omega_i,         # Idler Rabi Frequency (Related to electric field strength)
        Omega_I,         # Pump 1 Rabi Frequency (Related to electric field strength)
        Omega_II,        # Pump 2 Rabi Frequency (Related to electric field strength)
        Delta_s,         # Signal detuning
        Delta_i,         # Idler detuning
        Delta_I,         # Pump 1 detuning
        Delta_II,        # Pump 2 detuning
        temp,            # Temperature
        cell_length,     # Length of vacuum cell (determines number of steps)
        step_size = 0.01 # Distance covered by each iteration
    ):

    distance_covered = 0
    while distance_covered < cell_length:
        rho = solve_for_steady_state_doppler_broadened_density_matrix(
            Omega_s, Omega_i, Omega_I, Omega_II,
            Delta_s, Delta_i, Delta_I, Delta_II,
            temp
        )

        # Update Omega_s and Omega_i via the Maxwell-Bloch equations
        Omega_s, Omega_i = Maxwell_Bloch_step(rho, Omega_s, Omega_i, Delta_s, Delta_i, temp, step_size)

        # Update the distance covered to bring the loop closer to finishing
        distance_covered += step_size
    
    # Obtain the final density matrix with respect to the final values of Omega_s and Omega_i
    rho = solve_for_steady_state_doppler_broadened_density_matrix(
        Omega_s, Omega_i, Omega_I, Omega_II,
        Delta_s, Delta_i, Delta_I, Delta_II,
        temp
    )

    return rho, Omega_s, Omega_i



In [389]:
Delta = 0.01
rho, Omega_s, Omega_i = iterative_solve(
    0.001, 0.01, 0.1, 0.1,
    Delta, Delta, Delta, Delta,
    80, 0.05, 0.01
)

print(Omega_s)
print(Omega_i)


(0.001-6.320833960679134e-28j)
(0.01-9.993910641638508e-31j)


In [390]:
class parameterization:
    def __init__(
            self,
            cost,
            Omega_s, Omega_i, Omega_I, Omega_II,
            Delta_s, Delta_i, Delta_I, Delta_II,
            temp
        ):
        self.cost = cost

        self.Omega_s = Omega_s
        self.Omega_i = Omega_i
        self.Omega_I = Omega_I
        self.Omega_II = Omega_II

        self.Delta_s = Delta_s
        self.Delta_i = Delta_i
        self.Delta_I = Delta_I
        self.Delta_II = Delta_II

        self.temp = temp
    
    def print(self):
        print("Cost Function Value =", self.cost)
        print("Omega_s  =", self.Omega_s)
        print("Omega_i  =", self.Omega_i)
        print("Omega_I  =", self.Omega_i)
        print("Omega_II =", self.Omega_II)
        print("Delta_s  =", self.Delta_s)
        print("Delta_i  =", self.Delta_i)
        print("Delta_I  =", self.Delta_I)
        print("Delta_II =", self.Delta_II)
        print("temp     =", self.temp)

In [391]:
##################
# COST FUNCTIONS #
##################

# Super simple cost function to start with
def max_Omega_i(_curr_rho, _curr_Omega_s, curr_Omega_i):
    return abs(curr_Omega_i)

In [392]:
####################
# HELPER FUNCTIONS #
####################

def update_parameter_ranges(
        optimal_params: parameterization,
        Omega_s_range:  tuple[float, float],
        Omega_i_range:  tuple[float, float],
        Omega_I_range:  tuple[float, float],
        Omega_II_range: tuple[float, float],
        Delta_s_range:  tuple[float, float],
        Delta_i_range:  tuple[float, float],
        Delta_I_range:  tuple[float, float],
        temp_range:     tuple[float, float],
        divisor
    ):
    # Update parameter ranges
    new_Omega_s_halfwidth = (Omega_s_range[1] - Omega_s_range[0]) / (2*divisor)
    Omega_s_range = (optimal_params.Omega_s - new_Omega_s_halfwidth, optimal_params.Omega_s + new_Omega_s_halfwidth)

    new_Omega_i_halfwidth = (Omega_i_range[1] - Omega_i_range[0]) / (2*divisor)
    Omega_i_range = (optimal_params.Omega_i - new_Omega_i_halfwidth, optimal_params.Omega_i + new_Omega_i_halfwidth)

    new_Omega_I_halfwidth = (Omega_I_range[1] - Omega_I_range[0]) / (2*divisor)
    Omega_I_range = (optimal_params.Omega_I - new_Omega_I_halfwidth, optimal_params.Omega_I + new_Omega_I_halfwidth)

    new_Omega_II_halfwidth = (Omega_II_range[1] - Omega_II_range[0]) / (2*divisor)
    Omega_II_range = (optimal_params.Omega_II - new_Omega_II_halfwidth, optimal_params.Omega_II + new_Omega_II_halfwidth)

    new_Delta_s_halfwidth = (Delta_s_range[1] - Delta_s_range[0]) / (2*divisor)
    Delta_s_range = (optimal_params.Delta_s - new_Delta_s_halfwidth, optimal_params.Delta_s + new_Delta_s_halfwidth)

    new_Delta_i_halfwidth = (Delta_i_range[1] - Delta_i_range[0]) / (2*divisor)
    Delta_i_range = (optimal_params.Delta_i - new_Delta_i_halfwidth, optimal_params.Delta_i + new_Delta_i_halfwidth)

    new_Delta_I_halfwidth = (Delta_I_range[1] - Delta_I_range[0]) / (2*divisor)
    Delta_I_range = (optimal_params.Delta_I - new_Delta_I_halfwidth, optimal_params.Delta_I + new_Delta_I_halfwidth)

    new_temp_halfwidth = (temp_range[1] - temp_range[0]) / (2*divisor)
    temp_range = (optimal_params.temp - new_temp_halfwidth, optimal_params.temp + new_temp_halfwidth)

    return Omega_s_range, Omega_i_range, Omega_I_range, Omega_II_range, Delta_s_range, Delta_i_range, Delta_I_range, temp_range

In [ ]:
##########################
# OPTIMIZATION ALGORITHM #
##########################


# Use a random iterative Monte-Carlo algorithm:
#
# 1. Solve over randomly sampled points in multidimensional problem space
#
# 2. Take the solution which maximizes the cost function and use it as a center point
#    for another round of step 1 where each range has been significantly skrunk
#
# 3. Repeat 1 and 2 until a desired depth and return the corresponding
#    parameterization and solution
#
# TODO - Do we want to do function minimization of maximization????
# NOTE - The code will change depending on which is chosen
def find_optimal_params(
        Omega_s_range:  tuple[float, float],
        Omega_i_range:  tuple[float, float],
        Omega_I_range:  tuple[float, float],
        Omega_II_range: tuple[float, float],
        Delta_s_range:  tuple[float, float],
        Delta_i_range:  tuple[float, float],
        Delta_I_range:  tuple[float, float],
        # No Delta_II due to imposed conservation (see comment below)
        temp_range:     tuple[float, float],
        cost_function   = max_Omega_i, # Relevant cost function
        point_array     = [10, 10, 5], # Number of iterations at each level, stored as list # TODO - Make default much larger!!!
        divisor         = 4            # How much each range is shrunk upon iteration
    ):

    # Perform the iterative shrinking of the sample space `len(point_array)` times
    for level in range(len(point_array)):

        # Use a random point generation method rather than nested loops
        # to control how many points are sampled over and avoid
        # exloding time complexity
        #
        # NOTE: * is the splat operator which unpacks the tuple as function inputs
        Omega_s_vals = [random.uniform(*Omega_s_range) for _ in range(point_array[level])]
        Omega_i_vals = [random.uniform(*Omega_i_range) for _ in range(point_array[level])]
        Omega_I_vals = [random.uniform(*Omega_I_range) for _ in range(point_array[level])]
        Omega_II_vals = [random.uniform(*Omega_II_range) for _ in range(point_array[level])]

        Delta_s_vals = [random.uniform(*Delta_s_range) for _ in range(point_array[level])]
        Delta_i_vals = [random.uniform(*Delta_i_range) for _ in range(point_array[level])]
        Delta_I_vals = [random.uniform(*Delta_I_range) for _ in range(point_array[level])]

        # Since omega_s + omega_II = omega_i + omega_I and
        # omega_13 + omega_34 = omega_12 + omega_24 (little omega = frequency)
        # we must have Delta_s + Delta_II = Delta_i + Delta_I enforced.
        # So determine 3 parameters then calculate the fourth via this constraint
        Delta_II_vals = [(I+i-s) for (s,i,I) in zip(Delta_s_vals, Delta_i_vals, Delta_I_vals)]
        
        temp_vals = [random.uniform(*temp_range) for _ in range(point_array[level])]

        # Store all parameterizations along with their corresponding cost function values
        solutions = []


        for (Omega_s, Omega_i, Omega_I, Omega_II, Delta_s, Delta_i, Delta_I, Delta_II, temp) \
            in zip(
                Omega_s_vals, Omega_i_vals, Omega_I_vals, Omega_II_vals, 
                Delta_s_vals, Delta_i_vals, Delta_I_vals, Delta_II_vals,
                temp_vals
            ):

            # Solve the system using Master equation into Maxwell-Bloch, back into Master, and so on
            curr_rho, curr_Omega_s, curr_Omega_i = iterative_solve(
                Omega_s, Omega_i, Omega_I, Omega_II,
                Delta_s, Delta_i, Delta_I, Delta_II,
                temp, 0.05, 0.01 # TODO - Update cell length and step size
            )

            # Calculate cost function (NOTE - cost_function is an input parameter)
            cost = cost_function(curr_rho, curr_Omega_s, curr_Omega_i)

            # Add the current parameterization onto the list
            solutions.append(
                parameterization(cost, Omega_s, Omega_i, Omega_I, Omega_II, Delta_s, Delta_i, Delta_I, Delta_II, temp)
            )
        
        # Find maximizing parameterization out of current parameterizations
        optimal_params = solutions[0]
        for sol in solutions:
            if sol.cost > optimal_params.cost:
                optimal_params = sol
            
        # Update parameter ranges (shrink the sampling space)
        Omega_s_range, Omega_i_range, Omega_I_range, Omega_II_range, \
            Delta_s_range, Delta_i_range, Delta_I_range, temp_range = \
                update_parameter_ranges(
                    optimal_params,
                    Omega_s_range, Omega_i_range, Omega_I_range, Omega_II_range,
                    Delta_s_range, Delta_i_range, Delta_I_range,
                    temp_range,
                    divisor
                )
    
    # Return the final optimal parameterization
    return optimal_params


In [394]:
optimal_params = find_optimal_params(
    (0, 1),
    (0, 1),
    (0, 1),
    (0, 1),
    (0, 1),
    (0, 1),
    (0, 1),
    (0, 1)
)

optimal_params.print()

Cost Function Value = 1.0683543348288038
Omega_s  = 0.4045226942030093
Omega_i  = 1.0683543348288038
Omega_I  = 1.0683543348288038
Omega_II = 0.28540765764236853
Delta_s  = 0.4531012141974354
Delta_i  = 0.45727969646005606
Delta_I  = 0.15087218631024946
Delta_II = 0.15505066857287014
temp     = 0.6620492831952737


In [395]:
# num_vals = 500
# Delta_vals = np.linspace(-10.0, 10.0, num_vals)
# a_num = []
# b_num = []

# for Delta in Delta_vals:
#     H_total = form_full_Hamiltonian(0.001, 0.001, 0.1, 0.1, Delta, Delta, Delta, Delta)
#     rho = steadystate(H_total, c_ops, method='direct')
#     a_num.append(expect(num_a_op, rho))
#     b_num.append(expect(num_b_op, rho))

# plt.plot(Delta_vals, b_num, color='orange')
# plt.plot(Delta_vals, a_num, color='blue')
# plt.legend(('b', 'a'))